In [67]:
import pandas as pd
import re
import json

pd.set_option('display.max_colwidth', None)

# Carregamento das bases

Durante a tentativa de ingestão inicial da base conversas.json, o interpretador retornou erros de parsing (JSONDecodeError). Ao investigar o arquivo bruto, identifiquei que o campo de texto livre (mensagens dos usuários) continha caracteres de escape inválidos, como aspas duplas internas e quebras de linha (\n) não tratadas na origem.

<b>Solução utilizada</b>: Optei por criar um pré-processador utilizando Regez. Ele atua especificamente no payload de texto, higienizando os caracteres antes da leitura da tabela. Isso garantiu a retenção de 100% dos registros (46.960 linhas).

In [68]:
df_campanhas = pd.read_json('../data/campanhas.json')
# df_conversas = pd.read_json('../data/conversas.json', encoding='latin-1')
df_logs = pd.read_csv('../data/logs_omnichannel.csv')

### Saneamento do df conversas

In [69]:
with open('../data/conversas.json', 'r', encoding='latin-1') as file:
    raw = file.read()

def sanitize_text(match):
    content = match.group(1)
    
    # Transforma barras isoladas em barras duplas
    content = content.replace('\\', '\\\\')
    
    # Tratamento de aspas dentro das mensagens
    content = content.replace('"', "'")
    
    # Tratamento de quebras de linha
    content = content.replace('\n', ' ').replace('\r', '')
    
    # Remonta a chave mantendo a estrutura original
    return f'"text": "{content}",\n  "author"'

# Captura da mensagem e envio para a função de sanitização
cleaned_raw = re.sub(r'"text":\s*"(.*?)",\n\s*"author"', sanitize_text, raw, flags=re.DOTALL)

# Carregamento dos dados após limpeza
data = json.loads(cleaned_raw, strict=False)

df_conversas = pd.DataFrame(data)

# Amostra do resultado e check de volumetria
print("Total de registros carregados:", df_conversas.shape)
display(df_conversas.head())

Total de registros carregados: (46960, 10)


,session_id,text,author,user_id,publish_time,data,attributes,subscription_name,message_id,media_type
0,9e68a467-3063-4777-81ff-22afd7adf208,!,request,5511901234567,2026-03-20 11:31:06.420000 UTC,None,{},conversation-topic-sub-bq,18928201974961901,text
1,81070479-ac69-439f-a14b-369b8fb74b32,!,request,5511901234567,2026-03-20 20:32:40.392000 UTC,None,{},conversation-topic-sub-bq,18936509139188573,text
2,9e68a467-3063-4777-81ff-22afd7adf208,!,request,5511901234567,2026-03-20 11:31:42.460000 UTC,None,{},conversation-topic-sub-bq,18928682676616509,text
3,6d189cde-858b-45d9-b966-25b871dffd8e,!!!!,request,5511901234567,2026-03-20 16:11:47.983000 UTC,None,{},conversation-topic-sub-bq,18933458805543993,text
4,977ef681-2230-4836-9832-a6c16301e585,"\',\'\n https://especiais.magazineluiza.com.br/politica-de-privacidade/#:~:text\u003d47.960.950/0001%2D21-,%2C,-com%20sede%20na",request,5511901234567,2026-03-20 18:54:33.208000 UTC,None,{},conversation-topic-sub-bq,18935016322846916,text


# Investigação

Para validar o apontamento feito pela área de negócios, repliquei o comportamento do dashboard utilizando um LEFT JOIN entre as tabelas campanhas e conversas, utilizando o session_id como chave de relacionamento. O resultado confirmou a anomalia visualizada pelo CRM: zero disparos contabilizados para as campanhas relatadas.

In [70]:
print("--- Investigando Campanha Apple ---")

apple_camp = df_campanhas[(df_campanhas['publish_time'].dt.strftime('%Y-%m-%d') == '2026-03-19') & (df_campanhas['template'] == 'crm_cerebro_ads_apple_1903')]
display(apple_camp[['session_id', 'template', 'version', 'publish_time']].head())

print("\n--- Investigando Campanha Samsung ---")
samsung_camp = df_campanhas[(df_campanhas['publish_time'].dt.strftime('%Y-%m-%d') == '2026-03-20') & (df_campanhas['template'] == 'crm_cerebro_galaxys26')]
display(samsung_camp[['session_id', 'template', 'version', 'publish_time']].head())

# Simulação da query no dashboard
df_dashboard = pd.merge(df_campanhas, df_conversas, on='session_id', how='left', suffixes=('_campanha', '_conversa'))

disparos_apple = df_dashboard[df_dashboard['template'] == 'crm_cerebro_ads_apple_1903']['message_id_conversa'].count()
disparos_samsung = df_dashboard[df_dashboard['template'] == 'crm_cerebro_galaxys26']['message_id_conversa'].count()

print(f"\nDisparos da Apple contabilizados no painel: {disparos_apple}")
print(f"Disparos da Samsung contabilizados no painel: {disparos_samsung}")

--- Investigando Campanha Apple ---


,session_id,template,version,publish_time



--- Investigando Campanha Samsung ---


,session_id,template,version,publish_time



Disparos da Apple contabilizados no painel: 0
Disparos da Samsung contabilizados no painel: 0


### Busca e hipótese de parametrização

Como não foi retornado nenhum resultado, a hipótese levantada foi uma divergência de cadastro na tabela de dimensão campanhas.
Para realizar a varredura sem onerar processamento, apliquei primeiro um particionamento lógico por data (2026-03-19 e 2026-03-20), restringindo o volume de busca, e em seguida apliquei filtros textuais flexíveis.

<b>Conclusão</b>: A campanha para Apple foi disparada, mas salva no banco de dados com a nomenclatura e versão diferentes das informadas pelo analista. Já a campanha da Samsung nem chegou a ser salva na base de campanhas. Portanto, a próxima análise para a campanha da Samsung será verificar a base de logs do Omnichannel em busca do que aconteceu para os dados não serem salvos nas tabelas de campanhas e conversas.

In [71]:
print("--- Busca por Send Type (835 e 838) ---")
# Lista com variações possíveis
possible_send_types = ['sendtype-835', '835', 'sendtype-838', '838', 835, 838]

search_by_id = df_campanhas[
    (df_campanhas['publish_time'].dt.strftime('%Y-%m-%d').isin(['2026-03-19', '2026-03-20'])) & 
    (df_campanhas['version'].isin(possible_send_types))
]

display(search_by_id[['session_id', 'template', 'version', 'publish_time']])


print("\n--- Busca por palavras-chave ---")

search_by_name = df_campanhas[
    (df_campanhas['publish_time'].dt.strftime('%Y-%m-%d').isin(['2026-03-19', '2026-03-20'])) & 
    (df_campanhas['template'].str.contains('apple|galaxy|s26', na=False, case=False))
]
display(search_by_name[['session_id', 'template', 'version', 'publish_time']])

print('---  Escopo geral de campanhas e versões  ---')
full_scope = search_by_name[['template', 'version']].drop_duplicates()
display(full_scope)

# Verificar valores distintos contidos no resultado
display(search_by_name['template'].value_counts().to_frame())
display(search_by_name['version'].value_counts().to_frame())

--- Busca por Send Type (835 e 838) ---


,session_id,template,version,publish_time



--- Busca por palavras-chave ---


,session_id,template,version,publish_time
51,f7380de6-f819-4c9a-bfbf-c977819605b9,crm_cerebro_ads_apple_1003,1,2026-03-20 14:06:36.235000+00:00
243,2f0a37fd-1177-44a7-b915-f5b306164bca,crm_cerebro_ads_apple_1003,1,2026-03-20 10:09:25.569000+00:00
292,e6aa8a74-8799-46ab-a20b-72ac13dca96c,crm_cerebro_ads_apple_1003,1,2026-03-20 03:15:40.179000+00:00
293,48147423-f009-45f2-af79-96c36fc27223,crm_cerebro_ads_apple_1003,1,2026-03-20 14:51:02.329000+00:00
388,ad22a183-ed66-490e-a6fa-6547ae25b8c6,crm_cerebro_ads_apple_1003,1,2026-03-20 19:04:04.285000+00:00
...,...,...,...,...
11234,99aca927-36be-4195-82af-1d88c46beddd,crm_cerebro_ads_apple_1303,1,2026-03-19 18:41:56.070000+00:00
11291,fbce438d-c777-453c-87c6-f0b6754e6dd0,crm_cerebro_ads_apple_1003,1,2026-03-19 19:24:10.897000+00:00
11292,21f13334-aa9f-419d-a940-f83f2cfddb8e,crm_cerebro_ads_apple_1303,1,2026-03-19 20:54:22.583000+00:00
11345,7ccd7199-5646-46c8-adb2-f99d64ce9652,crm_cerebro_ads_apple_1303,1,2026-03-19 14:25:05.591000+00:00


---  Escopo geral de campanhas e versões  ---


,template,version
51,crm_cerebro_ads_apple_1003,1
389,crm_cerebro_ads_apple_1303,1
613,apple16e_natal_2025,sendtype-757
1717,apple7_natal_2025,sendtype-758
2111,crm_cerebro_ads_apple_at,1


,count
template,
crm_cerebro_ads_apple_1303,171
crm_cerebro_ads_apple_1003,66
apple7_natal_2025,11
apple16e_natal_2025,5
crm_cerebro_ads_apple_at,4


,count
version,
1,241
sendtype-758,11
sendtype-757,5


### Busca campanha Samsung

Na tabela de logs do provedor Omnichannel, utilizei a data principal do servidor de nuvem (timestamp) para garantir a otimização da busca cobrindo o fuso horário UTC (dias 20 e 21). No entanto, para a análise de negócio, exibi o jsonPayload.timestamp, que crava o momento exato em que a aplicação falhou.

<b>Conclusão</b>: O sistema do CRM enviou o botão de CTA em formato de texto plano em vez de um objeto estruturado. A API do provedor rejeitou o pacote (Erro: It is not a JSON type and cannot be deserialized), barrando a mensagem antes mesmo de iniciar o disparo. Dessa forma a mensagem nem chegou ao Whatsapp do cliente e por isso os dados de campanha não foram salvos nas tabelas.

In [66]:
print("---  Investigação Logs do Provedor (Omnichannel)  ---")

samsung_camp_errors = df_logs[
    (df_logs['timestamp'].str.contains('2026-03-20|2026-03-21', na=False)) & 
    (df_logs['jsonPayload.message'].str.contains('Comprar Galaxy S26', case=False, na=False))
]

print(f"Total de logs encontrados para a Samsung: {len(samsung_camp_errors)}")
display(samsung_camp_errors[['jsonPayload.message', 'jsonPayload.timestamp']])

print("\n--- Análise de Erros e Datas Distintas ---")

display(samsung_camp_errors['jsonPayload.message'].value_counts().to_frame(name='quantidade_erros'))

samsung_camp_errors['short_date'] = samsung_camp_errors['jsonPayload.timestamp'].str.slice(0, 11)

display(samsung_camp_errors['short_date'].value_counts().to_frame(name='quantidade_logs_por_dia'))

---  Investigação Logs do Provedor (Omnichannel)  ---
Total de logs encontrados para a Samsung: 40


,jsonPayload.message,jsonPayload.timestamp
981,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:56:22
1111,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:56:00
1132,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:55:56
1611,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:54:37
1625,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:54:32
1786,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:54:00
1800,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:53:57
1877,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:53:41
1972,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:53:25
2909,It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,'2026-03-21T02:49:59



--- Análise de Erros e Datas Distintas ---


,quantidade_erros
jsonPayload.message,
It is not a JSON type and cannot be deserialized: Comprar Galaxy S26,40


,quantidade_logs_por_dia
short_date,
'2026-03-21,40
